In [1]:
import os
import sys

sys.path.insert(0, "/Users/wenduoc/SpatialAgent/src")

In [2]:
from __future__ import annotations
from typing import Optional, Any, List
from dataclasses import dataclass

from openai import OpenAI
from openai import APIConnectionError, RateLimitError, OpenAIError


def _coerce_secret(v: Any) -> str:
    return v.get_secret_value() if hasattr(v, "get_secret_value") else v


def _extract_text(output: Any) -> str:
    if not output:
        return ""
    parts: List[str] = []
    for item in output:
        t = getattr(item, "type", None)
        if t == "message":
            for c in getattr(item, "content", []) or []:
                if getattr(c, "type", None) == "output_text":
                    txt = getattr(c, "text", None)
                    val = txt if isinstance(txt, str) else getattr(txt, "value", "")
                    if isinstance(val, str) and val.strip():
                        parts.append(val.strip())
        elif t == "output_text":
            txt = getattr(item, "text", None)
            if isinstance(txt, str) and txt.strip():
                parts.append(txt.strip())
            else:
                val = getattr(txt, "value", None)
                if isinstance(val, str) and val.strip():
                    parts.append(val.strip())
    return "\n\n".join(parts)


@dataclass
class OpenAIWebSearchEngine:
    api_key: Any
    default_model: str = "gpt-5"

    def __post_init__(self):
        key = _coerce_secret(self.api_key)
        if not key or str(key).startswith("*"):
            raise ValueError("Invalid OpenAI API key")
        self.client = OpenAI(api_key=key)

    def run(
        self,
        query: str,
        model: Optional[str] = None,
        max_output_tokens: int = 2000,
        instructions: Optional[str] = None,
    ) -> str:
        q = (query or "").strip()
        if not q:
            return "Error: empty query."

        mdl = model or self.default_model
        sys = (
            "You are a web research tool. Search the web and produce a concise answer with live URLs. "
            "Prefer official/primary sources; include the canonical link first. If ambiguous, list the top 2 options."
        )
        if instructions:
            sys += " " + instructions.strip()

        try:
            resp = self.client.responses.create(
                model=mdl,
                instructions=sys,
                input=q,
                tools=[{"type": "web_search"}],
                max_output_tokens=max_output_tokens,
            )
            # Fast path
            txt = getattr(resp, "output_text", None)
            if isinstance(txt, str) and txt.strip():
                return txt.strip()
            # Fallback parse across shapes
            txt2 = _extract_text(getattr(resp, "output", None))
            return txt2 if txt2 else "No output from web search."

        except RateLimitError as e:
            return f"OpenAI web search rate-limited: {e}"
        except APIConnectionError as e:
            return f"OpenAI web search connection error: {e}"
        except OpenAIError as e:
            return f"OpenAI web search error: {e}"
        except Exception as e:
            return f"Unexpected error during OpenAI web search: {e}"

In [3]:
key = os.environ.get("OPENAI_API_KEY", "")
engine = OpenAIWebSearchEngine(api_key=key, default_model="gpt-5")

q = "can you find a single cell reference dataset for developing human heart from czi cellxgene?."
out = engine.run(q)
print("\n=== OUTPUT ===\n")
print(out)


=== OUTPUT ===

No output from web search.


In [33]:
key = os.environ.get("OPENAI_API_KEY", "")
engine = OpenAIWebSearchEngine(api_key=key, default_model="gpt-5")

q = "aidan zhang  cmu."
out = engine.run(q)
print("\n=== OUTPUT ===\n")
print(out)


=== OUTPUT ===

OpenAI web search rate-limited: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5 in organization org-kv7LK9IREGMuXSAD7q2NekVi on tokens per min (TPM): Limit 30000, Used 30000, Requested 4885. Please try again in 9.77s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}


In [2]:
from agents.web_search_agent.tools_impl.openai_web_search_tool import (
    OpenAIWebSearchEngine,
)

In [3]:
from api_keys import APIKeys

In [4]:
key = os.environ.get("OPENAI_API_KEY", "")
engine = OpenAIWebSearchEngine(api_key=key, default_model="gpt-5")

q = "Find the lab website of jian ma from cmu."
out = engine.run(q)
print("\n=== OUTPUT ===\n")
print(out)

resp Response(id='resp_68a4be1b1874819b96bc8affc53b8492048a12b755e9fbc9', created_at=1755627035.0, error=None, incomplete_details=None, instructions='You are a web research tool. Search the web and produce a concise answer with live URLs. Prefer official/primary sources; include the canonical link first. If ambiguous, list the top 2 options.', metadata={}, model='gpt-5-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_68a4be1b784c819b87c37f301c67c158048a12b755e9fbc9', summary=[], type='reasoning', status=None), ResponseFunctionWebSearch(id='ws_68a4be1e7e98819bab40b7e5333bc085048a12b755e9fbc9', status='completed', type='web_search_call', action={'type': 'search', 'query': 'Jian Ma CMU lab website'}), ResponseReasoningItem(id='rs_68a4be2139e8819b9921dfd02519d17c048a12b755e9fbc9', summary=[], type='reasoning', status=None), ResponseOutputMessage(id='msg_68a4be2781c0819ba7ea34624359c31c048a12b755e9fbc9', content=[ResponseOutputText(annotations=[AnnotationURLCitation(end_

In [19]:
from agents.web_search_agent.tools import create_web_search_tools

In [22]:
keys = APIKeys(
    {
        "openai": os.environ.get("OPENAI_API_KEY", ""),
        "serp": os.environ.get("SERP_API_KEY", "") or "DUMMY",
        "email": "you@example.com",
    }
)
tools = create_web_search_tools(keys)

# find our tool by name
tool = next(t for t in tools if t.name == "openai_web_search_tool")
result = tool.invoke("Find the lab website of jian ma from cmu.")
print(result)

Jian Ma's lab at Carnegie Mellon University focuses on developing AI and machine learning methods to study the human genome and cellular organization, with significant implications for health and disease. Their research includes areas such as nuclear organization, single-cell epigenomics, spatial omics, and complex molecular interactions. ([cbd.cmu.edu](https://cbd.cmu.edu/people/ma.html?utm_source=openai))

You can find more information about the lab's research and members on their website: ([cs.cmu.edu](https://www.cs.cmu.edu/~jianma/?utm_source=openai)) 
